# 1. 라이브러리 로드

In [2]:
import pandas as pd

from datetime import datetime

from tqdm import tqdm

import sys
sys.path.append('C:/lucio 서강대 정보통신대학원/학위논문/아파트 거래량 이상탐지/apt_volume_anomaly_detection/dataset')

import warnings
warnings.filterwarnings('ignore')

# 2. 아파트 매매 거래량

In [3]:
# apt_volume_path = 'C:/lucio 서강대 정보통신대학원/학위논문/아파트 거래량 이상탐지/apt_volume_anomaly_detection/dataset/apt_volume/apt_purchase_volume_'

# # 초기값 설정
# start_year, start_month = 2010, 6
# end_year, end_month = 2024, 6

# # 1년 단위 파일명 리스트 컴프리헨션 생성
# date_index_lst = [
#     f"{year:04}{start_month:02}_{year+1:04}{start_month-1:02}"
#     for year in range(start_year, end_year)
# ]

# # 마지막 파일 추가 (202406_202412)
# date_index_lst.append("202406_202412")

In [4]:
# # 행정구 / 월 단위 데이터셋 생성
# monthly_apt_volume_df_lst = []
# for date_index in tqdm(date_index_lst):

#     origin_apt_volume_df = pd.read_excel(f'C:/lucio 서강대 정보통신대학원/학위논문/아파트 거래량 이상탐지/apt_volume_anomaly_detection/dataset/apt_volume/apt_purchase_volume_{date_index}.xlsx', header=12)

#     origin_apt_volume_df['region'] = origin_apt_volume_df['시군구'].map(lambda x: ' '.join(x.split(' ')[:2]))
#     origin_apt_volume_df['거래금액(만원)'] = origin_apt_volume_df['거래금액(만원)'].map(lambda x: int(x.replace(',', '')) * 10000)

#     single_monthly_apt_volume_df = origin_apt_volume_df.groupby(['region', '계약년월']).agg(
#                                                                                             volume_cnt=('NO', 'nunique'),
#                                                                                             avg_area_size=('전용면적(㎡)', 'mean'),
#                                                                                             avg_sales=('거래금액(만원)', 'mean'),
#                                                                                             avg_floor_cnt=('층', 'mean')
#                                                                                         ).reset_index().rename(columns={'계약년월' : 'month'})

#     monthly_apt_volume_df_lst.append(single_monthly_apt_volume_df)


# monthly_apt_volume_df = pd.concat(monthly_apt_volume_df_lst, ignore_index=True)

# # data type 변경
# monthly_apt_volume_df['date'] = pd.to_datetime(monthly_apt_volume_df['month'].astype(str), format='%Y%m')

In [5]:
# monthly_apt_volume_df.to_excel('C:/lucio 서강대 정보통신대학원/학위논문/아파트 거래량 이상탐지/apt_volume_anomaly_detection/dataset/apt_volume/apt_purchase_volume_monthly.xlsx')

In [6]:
monthly_apt_volume_df = pd.read_excel('dataset/apt_volume/apt_purchase_volume_monthly.xlsx', index_col=0)
monthly_apt_volume_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4375 entries, 0 to 4374
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   region         4375 non-null   object        
 1   month          4375 non-null   int64         
 2   volume_cnt     4375 non-null   int64         
 3   avg_area_size  4375 non-null   float64       
 4   avg_sales      4375 non-null   float64       
 5   avg_floor_cnt  4375 non-null   float64       
 6   date           4375 non-null   datetime64[ns]
dtypes: datetime64[ns](1), float64(3), int64(2), object(1)
memory usage: 273.4+ KB


# 3. 환율

In [7]:
exchange_rate_df = pd.read_excel('dataset/gov_index/exchange_rate.xlsx')

# 절상율 x, 환율 데이터만 get
exchange_rate_df = exchange_rate_df[~exchange_rate_df['Unnamed: 0'].isnull()].drop('Unnamed: 1', axis=1).T.iloc[1:, [0, 2]]
exchange_rate_df.columns = ['us_exchange_rate', 'jp_exchange_rate']

# date type 변환
exchange_rate_df = exchange_rate_df.reset_index().rename(columns={'index':'date'})
exchange_rate_df['date'] = pd.to_datetime(exchange_rate_df['date'].map(lambda x: str(x[:6])), format='%Y%m')

exchange_rate_df

,date,us_exchange_rate,jp_exchange_rate
0,2010-06-01,"1,222.2","1,380.6"
1,2010-07-01,"1,182.7","1,368.7"
2,2010-08-01,"1,198.1","1,423.8"
3,2010-09-01,"1,140.2","1,368.1"
4,2010-10-01,"1,125.3","1,394.9"
...,...,...,...
170,2024-08-01,"1,336.0",921.2
171,2024-09-01,"1,307.8",922.4
172,2024-10-01,"1,379.9",903.6
173,2024-11-01,"1,394.7",929.4


In [8]:
monthly_apt_volume_df = monthly_apt_volume_df.merge(exchange_rate_df, how='left', on='date')

In [9]:
monthly_apt_volume_df

,region,month,volume_cnt,avg_area_size,avg_sales,avg_floor_cnt,date,us_exchange_rate,jp_exchange_rate
0,서울특별시 강남구,201006,180,72.872581,7.497848e+08,9.033333,2010-06-01,"1,222.2","1,380.6"
1,서울특별시 강남구,201007,176,74.752961,7.591279e+08,8.482955,2010-07-01,"1,182.7","1,368.7"
2,서울특별시 강남구,201008,203,79.500165,8.385077e+08,7.758621,2010-08-01,"1,198.1","1,423.8"
3,서울특별시 강남구,201009,223,73.994781,7.879408e+08,7.896861,2010-09-01,"1,140.2","1,368.1"
4,서울특별시 강남구,201010,425,80.109369,8.605932e+08,8.214118,2010-10-01,"1,125.3","1,394.9"
...,...,...,...,...,...,...,...,...,...
4370,서울특별시 중랑구,202408,201,70.899098,6.694378e+08,10.174129,2024-08-01,"1,336.0",921.2
4371,서울특별시 중랑구,202409,96,69.455530,6.258438e+08,9.562500,2024-09-01,"1,307.8",922.4
4372,서울특별시 중랑구,202410,115,69.629382,6.276126e+08,9.800000,2024-10-01,"1,379.9",903.6
4373,서울특별시 중랑구,202411,90,68.257834,6.161556e+08,9.488889,2024-11-01,"1,394.7",929.4


# 4. 시장 금리

In [10]:
interest_rate_df = pd.read_excel('dataset/gov_index/interest_rate.xlsx').T

interest_rate_df.columns = interest_rate_df.iloc[0, :]
interest_rate_df = interest_rate_df.iloc[1:]
interest_rate_df = interest_rate_df.reset_index().rename(columns={'index':'date'})

interest_rate_df['date'] = pd.to_datetime(interest_rate_df['date'].map(lambda x: x[:6]), format='%Y%m')

interest_rate_df = interest_rate_df[['date', '기준금리']].rename(columns={'기준금리':'interest_rate'})
interest_rate_df

Unnamed: 0,date,interest_rate
0,2010-06-01,2.0
1,2010-07-01,2.25
2,2010-08-01,2.25
3,2010-09-01,2.25
4,2010-10-01,2.25
...,...,...
170,2024-08-01,3.5
171,2024-09-01,3.5
172,2024-10-01,3.25
173,2024-11-01,3.0


In [11]:
monthly_apt_volume_df = monthly_apt_volume_df.merge(interest_rate_df, how='left', on='date')

In [12]:
monthly_apt_volume_df

,region,month,volume_cnt,avg_area_size,avg_sales,avg_floor_cnt,date,us_exchange_rate,jp_exchange_rate,interest_rate
0,서울특별시 강남구,201006,180,72.872581,7.497848e+08,9.033333,2010-06-01,"1,222.2","1,380.6",2.0
1,서울특별시 강남구,201007,176,74.752961,7.591279e+08,8.482955,2010-07-01,"1,182.7","1,368.7",2.25
2,서울특별시 강남구,201008,203,79.500165,8.385077e+08,7.758621,2010-08-01,"1,198.1","1,423.8",2.25
3,서울특별시 강남구,201009,223,73.994781,7.879408e+08,7.896861,2010-09-01,"1,140.2","1,368.1",2.25
4,서울특별시 강남구,201010,425,80.109369,8.605932e+08,8.214118,2010-10-01,"1,125.3","1,394.9",2.25
...,...,...,...,...,...,...,...,...,...,...
4370,서울특별시 중랑구,202408,201,70.899098,6.694378e+08,10.174129,2024-08-01,"1,336.0",921.2,3.5
4371,서울특별시 중랑구,202409,96,69.455530,6.258438e+08,9.562500,2024-09-01,"1,307.8",922.4,3.5
4372,서울특별시 중랑구,202410,115,69.629382,6.276126e+08,9.800000,2024-10-01,"1,379.9",903.6,3.25
4373,서울특별시 중랑구,202411,90,68.257834,6.161556e+08,9.488889,2024-11-01,"1,394.7",929.4,3.0


# 5. 경제심리지수 & 소비자물가지수

In [13]:
esi_index_df = pd.read_excel('dataset/kosis/경제심리지수_201006_202412.xlsx').rename(columns={'시점':'date', '경제심리지수(원계열)':'esi_index'})
esi_index_df = esi_index_df.drop('경제심리지수(순환변동치)', axis=1)
esi_index_df['date'] = pd.to_datetime(esi_index_df['date'].map(lambda x: str(f"{x:.2f}").replace('.', '')), format='%Y%m')

esi_index_df

,date,esi_index
0,2010-06-01,115.5
1,2010-07-01,115.8
2,2010-08-01,114.2
3,2010-09-01,114.5
4,2010-10-01,112.3
...,...,...
170,2024-08-01,94.5
171,2024-09-01,94.0
172,2024-10-01,92.8
173,2024-11-01,93.0


In [14]:
cpi_index_df = pd.read_excel('dataset/kosis/소비자물가지수_201006_202412.xlsx').rename(columns={'시점':'date', '서울특별시':'cpi_index'}).drop_duplicates()
cpi_index_df['date'] = pd.to_datetime(cpi_index_df['date'].map(lambda x: str(f"{x:.2f}").replace('.', '')), format='%Y%m')

cpi_index_df

,date,cpi_index
0,2010-06-01,84.736
1,2010-07-01,84.991
2,2010-08-01,85.417
3,2010-09-01,86.183
4,2010-10-01,86.183
...,...,...
170,2024-08-01,113.940
171,2024-09-01,113.920
172,2024-10-01,114.040
173,2024-11-01,113.730


In [15]:
monthly_apt_volume_df = monthly_apt_volume_df.merge(esi_index_df, on='date').merge(cpi_index_df, on='date')
monthly_apt_volume_df

,region,month,volume_cnt,avg_area_size,avg_sales,avg_floor_cnt,date,us_exchange_rate,jp_exchange_rate,interest_rate,esi_index,cpi_index
0,서울특별시 강남구,201006,180,72.872581,7.497848e+08,9.033333,2010-06-01,"1,222.2","1,380.6",2.0,115.5,84.736
1,서울특별시 강남구,201007,176,74.752961,7.591279e+08,8.482955,2010-07-01,"1,182.7","1,368.7",2.25,115.8,84.991
2,서울특별시 강남구,201008,203,79.500165,8.385077e+08,7.758621,2010-08-01,"1,198.1","1,423.8",2.25,114.2,85.417
3,서울특별시 강남구,201009,223,73.994781,7.879408e+08,7.896861,2010-09-01,"1,140.2","1,368.1",2.25,114.5,86.183
4,서울특별시 강남구,201010,425,80.109369,8.605932e+08,8.214118,2010-10-01,"1,125.3","1,394.9",2.25,112.3,86.183
...,...,...,...,...,...,...,...,...,...,...,...,...
4370,서울특별시 중랑구,202408,201,70.899098,6.694378e+08,10.174129,2024-08-01,"1,336.0",921.2,3.5,94.5,113.940
4371,서울특별시 중랑구,202409,96,69.455530,6.258438e+08,9.562500,2024-09-01,"1,307.8",922.4,3.5,94.0,113.920
4372,서울특별시 중랑구,202410,115,69.629382,6.276126e+08,9.800000,2024-10-01,"1,379.9",903.6,3.25,92.8,114.040
4373,서울특별시 중랑구,202411,90,68.257834,6.161556e+08,9.488889,2024-11-01,"1,394.7",929.4,3.0,93.0,113.730


# 6. 인구

## 6.1 시군구별 이동자 수

In [16]:
pop_move_df = pd.read_csv('dataset/kosis/시군구별_이동자수_201006_202412.csv', encoding='cp949').drop('Unnamed: 10', axis=1).rename(columns={'시점':'date', '행정구역(시군구)별':'region'})
pop_move_df['date'] = pd.to_datetime(pop_move_df['date'].map(lambda x: x[:7].replace('.', '')), format='%Y%m')

pop_move_df

,region,date,총전입[명],총전출[명],순이동[명],시도내이동-시군구내[명],시도내이동-시군구간 전입[명],시도내이동-시군구간 전출[명],시도간전입[명],시도간전출[명]
0,종로구,2010-06-01,2225,2539,-314,471,1115,1364,639,704
1,종로구,2010-07-01,2098,2518,-420,397,1035,1366,666,755
2,종로구,2010-08-01,2346,2661,-315,366,1179,1509,801,786
3,종로구,2010-09-01,1962,2313,-351,444,969,1216,549,653
4,종로구,2010-10-01,2463,2832,-369,554,1196,1523,713,755
...,...,...,...,...,...,...,...,...,...,...
4370,강동구,2024-08-01,6405,5120,1285,2002,2471,1240,1932,1878
4371,강동구,2024-09-01,4919,4705,214,1693,1749,1261,1477,1751
4372,강동구,2024-10-01,6406,5704,702,2120,2405,1580,1881,2004
4373,강동구,2024-11-01,7254,4655,2599,2016,3293,1083,1945,1556


In [17]:
monthly_apt_volume_df['region'] = monthly_apt_volume_df['region'].map(lambda x: x.replace('서울특별시', '').strip())
monthly_apt_volume_df

,region,month,volume_cnt,avg_area_size,avg_sales,avg_floor_cnt,date,us_exchange_rate,jp_exchange_rate,interest_rate,esi_index,cpi_index
0,강남구,201006,180,72.872581,7.497848e+08,9.033333,2010-06-01,"1,222.2","1,380.6",2.0,115.5,84.736
1,강남구,201007,176,74.752961,7.591279e+08,8.482955,2010-07-01,"1,182.7","1,368.7",2.25,115.8,84.991
2,강남구,201008,203,79.500165,8.385077e+08,7.758621,2010-08-01,"1,198.1","1,423.8",2.25,114.2,85.417
3,강남구,201009,223,73.994781,7.879408e+08,7.896861,2010-09-01,"1,140.2","1,368.1",2.25,114.5,86.183
4,강남구,201010,425,80.109369,8.605932e+08,8.214118,2010-10-01,"1,125.3","1,394.9",2.25,112.3,86.183
...,...,...,...,...,...,...,...,...,...,...,...,...
4370,중랑구,202408,201,70.899098,6.694378e+08,10.174129,2024-08-01,"1,336.0",921.2,3.5,94.5,113.940
4371,중랑구,202409,96,69.455530,6.258438e+08,9.562500,2024-09-01,"1,307.8",922.4,3.5,94.0,113.920
4372,중랑구,202410,115,69.629382,6.276126e+08,9.800000,2024-10-01,"1,379.9",903.6,3.25,92.8,114.040
4373,중랑구,202411,90,68.257834,6.161556e+08,9.488889,2024-11-01,"1,394.7",929.4,3.0,93.0,113.730


In [18]:
monthly_apt_volume_df = monthly_apt_volume_df.merge(pop_move_df, on=['region', 'date'])
monthly_apt_volume_df

,region,month,volume_cnt,avg_area_size,avg_sales,avg_floor_cnt,date,us_exchange_rate,jp_exchange_rate,interest_rate,esi_index,cpi_index,총전입[명],총전출[명],순이동[명],시도내이동-시군구내[명],시도내이동-시군구간 전입[명],시도내이동-시군구간 전출[명],시도간전입[명],시도간전출[명]
0,강남구,201006,180,72.872581,7.497848e+08,9.033333,2010-06-01,"1,222.2","1,380.6",2.0,115.5,84.736,8674,8785,-111,2518,3229,3248,2927,3019
1,강남구,201007,176,74.752961,7.591279e+08,8.482955,2010-07-01,"1,182.7","1,368.7",2.25,115.8,84.991,8535,8873,-338,2559,3067,3139,2909,3175
2,강남구,201008,203,79.500165,8.385077e+08,7.758621,2010-08-01,"1,198.1","1,423.8",2.25,114.2,85.417,8943,9273,-330,2357,3463,3433,3123,3483
3,강남구,201009,223,73.994781,7.879408e+08,7.896861,2010-09-01,"1,140.2","1,368.1",2.25,114.5,86.183,6705,7435,-730,1864,2601,2858,2240,2713
4,강남구,201010,425,80.109369,8.605932e+08,8.214118,2010-10-01,"1,125.3","1,394.9",2.25,112.3,86.183,8268,8555,-287,2388,3243,3226,2637,2941
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4370,중랑구,202408,201,70.899098,6.694378e+08,10.174129,2024-08-01,"1,336.0",921.2,3.5,94.5,113.940,3762,3986,-224,1185,1560,1366,1017,1435
4371,중랑구,202409,96,69.455530,6.258438e+08,9.562500,2024-09-01,"1,307.8",922.4,3.5,94.0,113.920,3481,3676,-195,1169,1389,1203,923,1304
4372,중랑구,202410,115,69.629382,6.276126e+08,9.800000,2024-10-01,"1,379.9",903.6,3.25,92.8,114.040,3609,3971,-362,1225,1434,1310,950,1436
4373,중랑구,202411,90,68.257834,6.161556e+08,9.488889,2024-11-01,"1,394.7",929.4,3.0,93.0,113.730,3284,3555,-271,1099,1239,1188,946,1268


## 6.2 시군구별 1세단위 주민등록인구

In [19]:
# 초기값 설정
start_year, start_month = 2010, 6
end_year, end_month = 2024, 6

# 1년 단위 파일명 리스트 컴프리헨션 생성
date_index_lst = [
    f"{year:04}{start_month:02}_{year+1:04}{start_month-1:02}"
    for year in range(start_year, end_year)
]

# 마지막 파일 추가 (202406_202412)
date_index_lst.append("202406_202412")

In [20]:
# 나이 문자열을 정수형으로 변환하는 함수
def convert_age(age_str):
    if age_str == '100세 이상':
        return 100
    else:
        return int(age_str.replace('세', ''))

In [21]:
# 행정구 / 월 단위 데이터셋 생성
monthly_pop_df_lst = []
for date_index in tqdm(date_index_lst):

    sample_pop_df = pd.read_csv(f'dataset/kosis/시군구별_1세단위_주민등록인구_{date_index}.csv', encoding='cp949').drop('Unnamed: 6', axis=1)
    sample_pop_df.columns = ['region', 'age', 'date', 'tpop', 'mpop', 'wpop']

    sample_pop_df = sample_pop_df.query("age != '계'").reset_index(drop=True)

    # age 컬럼을 정수형으로 변환하여 새로운 컬럼 생성
    sample_pop_df['age_decade'] = sample_pop_df['age'].apply(convert_age)

    # 10세 단위 그룹으로 변환 (예: 0, 10, 20, ...)
    sample_pop_df['age_decade'] = (sample_pop_df['age_decade'] // 10) * 10

    # 옵션: 10세 단위 그룹을 문자열 레이블로 변환 (예: "0대", "10대", ...)
    sample_pop_df['age_decade'] = sample_pop_df['age_decade'].astype(str) + '대'

    sample_pop_df.loc[sample_pop_df['age_decade'] == '0대', 'age_decade'] = '10대 미만'
    sample_pop_df.loc[sample_pop_df['age_decade'] == '100대', 'age_decade'] = '90대 이상'

    sample_pop_df_pivot = sample_pop_df.pivot_table(
                                            index=['region', 'date'],
                                            columns=['age_decade'],
                                            values=['tpop', 'mpop', 'wpop'],
                                            aggfunc='sum'
                                        )

    sample_pop_df_pivot.columns = list(map("_".join, sample_pop_df_pivot.columns))
    sample_pop_df_pivot = sample_pop_df_pivot.reset_index()

    # 날짜 변환
    sample_pop_df_pivot['date'] = pd.to_datetime(sample_pop_df_pivot['date'].map(lambda x: x[:7].replace('.', '')), format='%Y%m')

    monthly_pop_df_lst.append(sample_pop_df_pivot)


monthly_pop_df = pd.concat(monthly_pop_df_lst, ignore_index=True)
monthly_pop_df

100%|██████████| 15/15 [00:02<00:00,  5.80it/s]


,region,date,mpop_10대,mpop_10대 미만,mpop_20대,mpop_30대,mpop_40대,mpop_50대,mpop_60대,mpop_70대,...,wpop_10대 미만,wpop_20대,wpop_30대,wpop_40대,wpop_50대,wpop_60대,wpop_70대,wpop_80대,wpop_90대,wpop_90대 이상
0,강남구,2010-06-01,43149,21459,43837,48192,46695,36892,21179,7403,...,20090,51569,53205,52560,40938,21354,9561,5053,805,26
1,강남구,2010-07-01,43163,21475,43675,48103,46709,36973,21221,7471,...,20113,51370,53239,52579,40998,21429,9580,5049,808,25
2,강남구,2010-08-01,43181,21446,43454,48022,46721,37107,21204,7524,...,20049,51175,53189,52627,41125,21455,9633,5048,823,24
3,강남구,2010-09-01,43080,21403,43311,47966,46601,37242,21207,7573,...,20040,50922,53175,52662,41277,21476,9643,5065,828,22
4,강남구,2010-10-01,43335,21464,43389,48713,47610,38197,21708,7832,...,20115,51022,53947,53445,42326,22132,9865,5244,959,97
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4370,중랑구,2024-08-01,12079,9263,23033,29628,27210,31567,30889,15817,...,8930,25437,28546,25025,32440,33038,18692,9266,1465,49
4371,중랑구,2024-09-01,12070,9192,22870,29651,27176,31498,30885,15901,...,8868,25332,28546,25019,32352,33121,18746,9325,1465,49
4372,중랑구,2024-10-01,12008,9151,22745,29667,27092,31415,30920,15994,...,8839,25207,28576,24985,32277,33117,18860,9344,1488,45
4373,중랑구,2024-11-01,11965,9082,22605,29694,27030,31355,30939,16053,...,8802,25136,28510,24951,32191,33114,18976,9365,1494,47


In [22]:
monthly_apt_volume_df = monthly_apt_volume_df.merge(monthly_pop_df, on=['region', 'date'])
monthly_apt_volume_df

,region,month,volume_cnt,avg_area_size,avg_sales,avg_floor_cnt,date,us_exchange_rate,jp_exchange_rate,interest_rate,...,wpop_10대 미만,wpop_20대,wpop_30대,wpop_40대,wpop_50대,wpop_60대,wpop_70대,wpop_80대,wpop_90대,wpop_90대 이상
0,강남구,201006,180,72.872581,7.497848e+08,9.033333,2010-06-01,"1,222.2","1,380.6",2.0,...,20090,51569,53205,52560,40938,21354,9561,5053,805,26
1,강남구,201007,176,74.752961,7.591279e+08,8.482955,2010-07-01,"1,182.7","1,368.7",2.25,...,20113,51370,53239,52579,40998,21429,9580,5049,808,25
2,강남구,201008,203,79.500165,8.385077e+08,7.758621,2010-08-01,"1,198.1","1,423.8",2.25,...,20049,51175,53189,52627,41125,21455,9633,5048,823,24
3,강남구,201009,223,73.994781,7.879408e+08,7.896861,2010-09-01,"1,140.2","1,368.1",2.25,...,20040,50922,53175,52662,41277,21476,9643,5065,828,22
4,강남구,201010,425,80.109369,8.605932e+08,8.214118,2010-10-01,"1,125.3","1,394.9",2.25,...,20115,51022,53947,53445,42326,22132,9865,5244,959,97
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4370,중랑구,202408,201,70.899098,6.694378e+08,10.174129,2024-08-01,"1,336.0",921.2,3.5,...,8930,25437,28546,25025,32440,33038,18692,9266,1465,49
4371,중랑구,202409,96,69.455530,6.258438e+08,9.562500,2024-09-01,"1,307.8",922.4,3.5,...,8868,25332,28546,25019,32352,33121,18746,9325,1465,49
4372,중랑구,202410,115,69.629382,6.276126e+08,9.800000,2024-10-01,"1,379.9",903.6,3.25,...,8839,25207,28576,24985,32277,33117,18860,9344,1488,45
4373,중랑구,202411,90,68.257834,6.161556e+08,9.488889,2024-11-01,"1,394.7",929.4,3.0,...,8802,25136,28510,24951,32191,33114,18976,9365,1494,47


# 7. 부동산 뉴스 데이터
* 감성 비율
* 2024년 10월 ~ 12월 결과 추가 필요

In [39]:
news_sentiment_df = pd.read_csv('dataset/naver_news/refined_news_topic_sentiment.csv', index_col=0)
news_sentiment_df = news_sentiment_df.groupby(['region', 'yyyymm']).agg(
                                                                        article_cnt=('article', 'nunique'),
                                                                        
                                                                        positive_ratio=('positive_ratio', 'mean'),
                                                                        negative_ratio=('negative_ratio', 'mean'),
                                                                        neutral_ratio=('neutral_ratio', 'mean')
                                                                    ).reset_index().rename(columns={'yyyymm':'date'})

news_sentiment_df['date'] = pd.to_datetime(news_sentiment_df['date'].map(lambda x: x.replace('-', '')[:6]), format='%Y%m')

news_sentiment_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3221 entries, 0 to 3220
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   region          3221 non-null   object        
 1   date            3221 non-null   datetime64[ns]
 2   article_cnt     3221 non-null   int64         
 3   positive_ratio  3219 non-null   float64       
 4   negative_ratio  3219 non-null   float64       
 5   neutral_ratio   3219 non-null   float64       
dtypes: datetime64[ns](1), float64(3), int64(1), object(1)
memory usage: 151.1+ KB


In [45]:
monthly_apt_volume_df = monthly_apt_volume_df.merge(news_sentiment_df, how='left', on=['region', 'date'])
monthly_apt_volume_df

,region,month,volume_cnt,avg_area_size,avg_sales,avg_floor_cnt,date,us_exchange_rate,jp_exchange_rate,interest_rate,...,wpop_50대,wpop_60대,wpop_70대,wpop_80대,wpop_90대,wpop_90대 이상,article_cnt,positive_ratio,negative_ratio,neutral_ratio
0,강남구,201006,180,72.872581,7.497848e+08,9.033333,2010-06-01,"1,222.2","1,380.6",2.0,...,40938,21354,9561,5053,805,26,16.0,0.631250,0.118750,0.250000
1,강남구,201007,176,74.752961,7.591279e+08,8.482955,2010-07-01,"1,182.7","1,368.7",2.25,...,40998,21429,9580,5049,808,25,4.0,0.362500,0.300000,0.337500
2,강남구,201008,203,79.500165,8.385077e+08,7.758621,2010-08-01,"1,198.1","1,423.8",2.25,...,41125,21455,9633,5048,823,24,18.0,0.377778,0.233333,0.388889
3,강남구,201009,223,73.994781,7.879408e+08,7.896861,2010-09-01,"1,140.2","1,368.1",2.25,...,41277,21476,9643,5065,828,22,3.0,0.183333,0.233333,0.583333
4,강남구,201010,425,80.109369,8.605932e+08,8.214118,2010-10-01,"1,125.3","1,394.9",2.25,...,42326,22132,9865,5244,959,97,3.0,0.550000,0.083333,0.366667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4370,중랑구,202408,201,70.899098,6.694378e+08,10.174129,2024-08-01,"1,336.0",921.2,3.5,...,32440,33038,18692,9266,1465,49,1.0,0.750000,0.050000,0.200000
4371,중랑구,202409,96,69.455530,6.258438e+08,9.562500,2024-09-01,"1,307.8",922.4,3.5,...,32352,33121,18746,9325,1465,49,3.0,0.566667,0.200000,0.233333
4372,중랑구,202410,115,69.629382,6.276126e+08,9.800000,2024-10-01,"1,379.9",903.6,3.25,...,32277,33117,18860,9344,1488,45,NaN,NaN,NaN,NaN
4373,중랑구,202411,90,68.257834,6.161556e+08,9.488889,2024-11-01,"1,394.7",929.4,3.0,...,32191,33114,18976,9365,1494,47,NaN,NaN,NaN,NaN


# 8. 아파트 전월세 거래량

In [59]:
qq = pd.read_excel(f'dataset/apt_rent_volume/apt_rent_volume_201106_201205.xlsx', header=12)
qq.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 147088 entries, 0 to 147087
Data columns (total 21 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   NO            147088 non-null  int64  
 1   시군구           147088 non-null  object 
 2   번지            147088 non-null  object 
 3   본번            147088 non-null  int64  
 4   부번            147088 non-null  int64  
 5   단지명           147088 non-null  object 
 6   전월세구분         147088 non-null  object 
 7   전용면적(㎡)       147088 non-null  float64
 8   계약년월          147088 non-null  int64  
 9   계약일           147088 non-null  int64  
 10  보증금(만원)       147088 non-null  object 
 11  월세금(만원)       147088 non-null  object 
 12  층             147086 non-null  float64
 13  건축년도          147088 non-null  int64  
 14  도로명           147088 non-null  object 
 15  계약기간          147088 non-null  object 
 16  계약구분          147088 non-null  object 
 17  갱신요구권 사용      147088 non-null  object 
 18  종전계약

In [66]:
print(qq['월세금(만원)'].dtype)

object


In [67]:
# 행정구 / 월 단위 데이터셋 생성
monthly_apt_rent_volume_df_lst = []
for date_index in tqdm(date_index_lst):

    origin_apt_rent_volume_df = pd.read_excel(f'dataset/apt_rent_volume/apt_rent_volume_{date_index}.xlsx', header=12)

    origin_apt_rent_volume_df['region'] = origin_apt_rent_volume_df['시군구'].map(lambda x: ' '.join(x.split(' ')[:2]))
    origin_apt_rent_volume_df['보증금(만원)'] = origin_apt_rent_volume_df['보증금(만원)'].map(lambda x: int(x.replace(',', '')) * 10000)

    if origin_apt_rent_volume_df['월세금(만원)'].dtype == 'object':
        origin_apt_rent_volume_df['월세금(만원)'] = origin_apt_rent_volume_df['월세금(만원)'].map(lambda x: int(x.replace(',', '')) * 10000)
    else:
        origin_apt_rent_volume_df['월세금(만원)'] = origin_apt_rent_volume_df['월세금(만원)'] * 10000

    single_monthly_apt_rent_volume_df = origin_apt_rent_volume_df.groupby(['region', '계약년월', '전월세구분']).agg(
                                                                                            rent_volume_cnt=('NO', 'nunique'),
                                                                                            avg_rent_area_size=('전용면적(㎡)', 'mean'),

                                                                                            avg_rent_deposit=('보증금(만원)', 'mean'),
                                                                                            avg_rent_monthly_deposit=('월세금(만원)', 'mean'),

                                                                                            avg_rent_floor_cnt=('층', 'mean')
                                                                                        ).reset_index().rename(columns={'계약년월' : 'date'})
    
    single_monthly_apt_rent_volume_df_pivot = single_monthly_apt_rent_volume_df.pivot_table(
                                                                                    index=['region', 'date'],
                                                                                    columns='전월세구분',
                                                                                    values=['rent_volume_cnt', 'avg_rent_area_size', 'avg_rent_deposit', 'avg_rent_monthly_deposit', 'avg_rent_floor_cnt'],
                                                                                    aggfunc={
                                                                                        'rent_volume_cnt':'sum', 'avg_rent_area_size':'mean', 
                                                                                        'avg_rent_deposit':'mean', 'avg_rent_monthly_deposit':'mean',
                                                                                        'avg_rent_floor_cnt':'mean'
                                                                                    }
                                                                                )
    single_monthly_apt_rent_volume_df_pivot.columns = list(map("_".join, single_monthly_apt_rent_volume_df_pivot.columns))
    single_monthly_apt_rent_volume_df_pivot = single_monthly_apt_rent_volume_df_pivot.reset_index()

    monthly_apt_rent_volume_df_lst.append(single_monthly_apt_rent_volume_df_pivot)


monthly_apt_rent_volume_df = pd.concat(monthly_apt_rent_volume_df_lst, ignore_index=True)

# data type 변경
monthly_apt_rent_volume_df['date'] = pd.to_datetime(monthly_apt_rent_volume_df['date'].astype(str), format='%Y%m')

monthly_apt_rent_volume_df

100%|██████████| 15/15 [58:59<00:00, 235.93s/it]


,region,date,avg_rent_area_size_월세,avg_rent_area_size_전세,avg_rent_deposit_월세,avg_rent_deposit_전세,avg_rent_floor_cnt_월세,avg_rent_floor_cnt_전세,avg_rent_monthly_deposit_월세,avg_rent_monthly_deposit_전세,rent_volume_cnt_월세,rent_volume_cnt_전세
0,서울특별시 강남구,2011-01-01,72.282738,84.558319,1.256026e+08,3.753957e+08,8.602564,7.972805,1.342468e+06,0.0,312,1287
1,서울특별시 강남구,2011-02-01,68.055820,84.003195,1.099265e+08,3.647988e+08,7.823529,7.653291,1.161601e+06,0.0,306,1246
2,서울특별시 강남구,2011-03-01,68.094070,79.907315,1.076309e+08,3.409725e+08,7.729231,7.732037,1.237138e+06,0.0,325,1183
3,서울특별시 강남구,2011-04-01,64.920287,81.555523,9.189286e+07,3.500034e+08,8.016807,7.505419,1.211555e+06,0.0,238,1015
4,서울특별시 강남구,2011-05-01,68.687800,85.655275,1.246616e+08,3.758372e+08,8.402542,7.896480,1.160169e+06,0.0,236,966
...,...,...,...,...,...,...,...,...,...,...,...,...
4195,서울특별시 중랑구,2024-08-01,43.403727,68.707353,9.178233e+07,3.180910e+08,8.525157,9.748848,4.574528e+05,0.0,318,434
4196,서울특별시 중랑구,2024-09-01,54.543694,70.775628,1.246193e+08,3.973028e+08,10.701987,10.180791,4.874503e+05,0.0,302,177
4197,서울특별시 중랑구,2024-10-01,45.535117,65.671968,1.038602e+08,3.796106e+08,9.392265,9.840426,5.877901e+05,0.0,181,188
4198,서울특별시 중랑구,2024-11-01,35.391442,69.296978,7.183326e+07,3.904101e+08,8.862661,9.398990,4.457082e+05,0.0,233,198


In [68]:
monthly_apt_rent_volume_df.to_excel('apt_rent_volume_monthly.xlsx')